# 03 — Comparison and the pre-registered decision

Paired match bootstrap (1000 draws, seed 20261108, 95 %). H1: `exp5_touch_ens3` − `exp3_ref_ens3`, provider labels, `target_like`: CI lower bound > 0 **and** mean Δ > `exp5_touch` seed spread. The ctrl file additionally needs `L_ctrl` non-inferiority (CI lower bound > −0.01). Everything else is report-only.

In [1]:
from pathlib import Path
import os, sys, json, time
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp5_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_5')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp5_common as C
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp1_common as C1
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp5_protocol as P, exp5_eval as E, exp5_report as R, exp5_diagnostics as G
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PREREG verified dc28ebfcacfa2676 | frozen 2026-11-10T09:06:10+00:00


In [2]:
reference = pd.read_csv(C.C3.RESULTS / 'reference_rows.csv')
reference = reference[reference.label_set.isin(C.S1_LABELS)]
rows = R.collect_rows([reference, pd.read_csv(C.RESULTS / 'argmax_rows.csv')])
rows.to_csv(C.RESULTS / 'per_match_results.csv', index=False)
table = R.summary_table(rows)
table.to_csv(C.RESULTS / 'comparison_table.csv', index=False)
cols = ['variant', 'mean_spell_f1', 'min_spell_f1', 'mean_macro_f1', 'mean_count_error', 'max_abs_count_error',
        'delta_vs_ref_ens3', 'ci95_low_vs_ref', 'ci95_high_vs_ref', 'wins_vs_ref', 'count_gate']
for label_set in C.S1_LABELS:
    for view in C.VIEWS:
        part = table[(table.label_set == label_set) & (table.view == view)]
        print(f'== {label_set} / {view}'); print(R.as_markdown(part, cols)); print()

== provider / original
| variant | mean_spell_f1 | min_spell_f1 | mean_macro_f1 | mean_count_error | max_abs_count_error | delta_vs_ref_ens3 | ci95_low_vs_ref | ci95_high_vs_ref | wins_vs_ref | count_gate |
|---|---|---|---|---|---|---|---|---|---|---|
| exp5_touch_ens3 | 0.676 | 0.623 | 0.923 | -0.201 | 0.317 | +0.010 | -0.000 | +0.020 | 4 | False |
| exp3_ref_ens3 | 0.666 | 0.621 | 0.922 | -0.204 | 0.360 | +0.000 | +0.000 | +0.000 | 0 | False |
| exp4_touch_ens3 | 0.681 | 0.628 | 0.924 | -0.218 | 0.348 | +0.015 | +0.006 | +0.025 | 6 | False |
| exp45_touch_ens6 | 0.680 | 0.625 | 0.924 | -0.209 | 0.317 | +0.014 | +0.005 | +0.024 | 6 | False |
| exp2_aug | 0.657 | 0.596 | 0.917 | -0.200 | 0.354 | -0.009 | -0.016 | -0.002 | 2 | False |
| exp5_touch | 0.671 | 0.611 | 0.922 | -0.201 | 0.307 | +0.005 | -0.004 | +0.012 | 4 | False |
| exp5_touch_s1 | 0.666 | 0.609 | 0.920 | -0.215 | 0.341 | +0.000 | -0.013 | +0.014 | 3 | False |
| exp5_touch_s2 | 0.665 | 0.617 | 0.919 | -0.188 | 0.297 | -0.

## Per match, `target_like` (spellF1, count error in brackets)

In [3]:
show = (C.PRIMARY, C.REFERENCE, 'exp4_touch_ens3', 'exp45_touch_ens6', *P.ARMS)
for label_set in C.DECISION_LABELS:
    print(f'== {label_set}'); print(R.per_match(rows, label_set, 'target_like', show).to_string()); print()

== provider
variant exp5_touch_ens3  exp3_ref_ens3 exp4_touch_ens3 exp45_touch_ens6     exp5_touch  exp5_touch_s1  exp5_touch_s2
match                                                                                                               
J03WMX    0.617 (-0.33)  0.615 (-0.37)   0.617 (-0.36)    0.618 (-0.32)  0.604 (-0.31)  0.608 (-0.33)  0.611 (-0.34)
J03WN1    0.653 (-0.26)  0.659 (-0.29)   0.665 (-0.24)     0.66 (-0.25)  0.651 (-0.23)  0.649 (-0.24)  0.641 (-0.25)
J03WOH    0.696 (-0.25)   0.67 (-0.23)   0.683 (-0.24)    0.692 (-0.25)  0.682 (-0.24)   0.68 (-0.26)  0.684 (-0.24)
J03WOY    0.692 (-0.16)  0.695 (-0.16)   0.707 (-0.19)    0.704 (-0.19)  0.693 (-0.19)  0.689 (-0.16)  0.676 (-0.15)
J03WPY    0.698 (-0.20)  0.691 (-0.18)   0.701 (-0.21)    0.701 (-0.21)  0.696 (-0.22)  0.686 (-0.22)  0.686 (-0.23)
J03WQQ    0.711 (-0.08)  0.698 (-0.06)   0.711 (-0.12)    0.716 (-0.08)  0.712 (-0.07)  0.712 (-0.12)  0.682 (-0.06)
J03WR9    0.654 (-0.15)  0.621 (-0.13)   0.671 (-0.1

## exp_0 quality gates (vs the same view's baseline), provider and `L_ctrl`

In [4]:
gates = R.gate_rows(rows, 'provider') + R.gate_rows(rows, 'ctrl')
print(pd.DataFrame([{'label_set': g['label_set'], 'view': g['view'], 'variant': g['variant'], **g['checks'], 'pass': g['pass'],
                     'worst_count_error': round(g['worst_count_error'], 3), 'worst_match': g['worst_count_match']}
                    for g in gates]).to_string(index=False))

label_set        view          variant  gain_at_least_005  ci_lower_above_zero  macro_regression_at_most_001  all_match_counts_within_30pct  pass  worst_count_error worst_match
 provider    original  exp5_touch_ens3               True                 True                          True                          False False              0.317      J03WMX
 provider    original    exp3_ref_ens3               True                 True                          True                          False False              0.360      J03WMX
 provider    original  exp4_touch_ens3               True                 True                          True                          False False              0.348      J03WMX
 provider    original exp45_touch_ens6               True                 True                          True                          False False              0.317      J03WMX
 provider    original         exp2_aug               True                 True                          True       

## Pre-registered decision (H1: CI clause and seed-spread clause), adoption tree

In [5]:
decision = R.decide(rows)
C1.write_json(C.RESULTS / 'decision.json', decision)
h1 = decision['H1']
d = h1['delta']
print(f"H1 {C.PRIMARY} - {C.REFERENCE}, provider / target_like: {d['mean_delta']:+.4f} [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}] wins {d['wins']}/7")
print('  CI clause  :', h1['ci_clause'])
print('  seed clause:', h1['seed_clause'])
print('H1', 'PASSED' if h1['passed'] else 'FAILED')
c = decision['ctrl_non_inferiority']
print(f"L_ctrl non-inferiority (ctrl file only): {c['mean_delta']:+.4f} [{c['ci95'][0]:+.4f}, {c['ci95'][1]:+.4f}] -> {'pass' if c['passed'] else 'fail'}")
print('adoption:', json.dumps(decision['adoption']))

H1 exp5_touch_ens3 - exp3_ref_ens3, provider / target_like: +0.0102 [+0.0009, +0.0199] wins 5/7
  CI clause  : {'rule': 'paired 95% CI lower bound > 0.0', 'ci_low': 0.0008911982458745619, 'passed': True}
  seed clause: {'rule': 'mean delta > exp5_touch seed spread (provider, target_like)', 'mean_delta': 0.010186519131984697, 'seed_spread': 0.009194151582759158, 'passed': True}
H1 PASSED
L_ctrl non-inferiority (ctrl file only): -0.0083 [-0.0204, +0.0041] -> fail
adoption: {"write_files": true, "variant": "exp5_touch_ens3", "members": ["exp5_touch", "exp5_touch_s1", "exp5_touch_s2"], "label_sets": ["provider"], "files": {"provider": "predictions/exp_5_candidate_provider.csv"}, "ctrl_non_inferiority": {"rule": "L_ctrl paired 95% CI lower bound > -0.01", "ci_low": -0.020444663059741765, "passed": false}, "statement": "H1 passed: provider candidate only; L_ctrl non-inferiority failed, exp_3's L_ctrl candidate stays", "configs": {"provider": {"temperature": 1.1147855174930903, "lambda_team":

## Replication (report only): exp_5 vs exp_4 touch, per cell; pooled 6-seed ensemble; single seeds

In [6]:
replication = pd.DataFrame(decision['report_only']['replication'])
replication.to_csv(C.RESULTS / 'replication_table.csv', index=False)
print(replication.round(4).to_string(index=False))
ro = decision['report_only']
for item in ro['single_seeds_vs_reference']:
    print(f"  {item['label_set']:8s} {item['view']:12s} {item['a']:14s} - {item['b']}: {item['mean_delta']:+.4f} [{item['ci95'][0]:+.4f}, {item['ci95'][1]:+.4f}] wins {item['wins']}")
print(pd.DataFrame([{k: v for k, v in s.items() if k != 'seeds'} | s['seeds'] for s in ro['seed_spreads']]).round(4).to_string(index=False))

label_set        view  exp5_delta  exp5_ci_low  exp5_ci_high  exp5_wins  exp4_delta  exp4_ci_low  exp4_ci_high  exp4_wins  exp5_minus_exp4  paired_ci_low  paired_ci_high  paired_wins  ens6_delta  ens6_ci_low  ens6_ci_high  ens6_wins
 provider    original      0.0099      -0.0004        0.0204          4      0.0146       0.0061        0.0249          6          -0.0047        -0.0104          0.0020            2      0.0137       0.0052        0.0242          6
 provider target_like      0.0102       0.0009        0.0199          5      0.0151       0.0069        0.0274          7          -0.0050        -0.0115          0.0019            1      0.0151       0.0064        0.0259          7
  track_1    original      0.0077      -0.0028        0.0197          4      0.0135       0.0073        0.0202          6          -0.0057        -0.0110          0.0015            1      0.0113       0.0004        0.0236          5
  track_1 target_like      0.0015      -0.0075        0.0105        

## Generalisation gap and short-possession recall (report only)

In [7]:
gap = pd.read_csv(C.RESULTS / 'gap_table.csv')
print(gap[gap.level == 'family'].round(4).to_string(index=False))
exp4_gap = G.exp4_gap_context()
print('exp_4 context (read in place):'); print(exp4_gap.round(4).to_string(index=False))
recall = pd.read_csv(C.RESULTS / 'short_recall.csv')
recall_summary = recall.groupby(['view', 'variant'])[['recall_short_decoded', 'recall_short_argmax', 'recall_all_decoded']].mean()
print(recall_summary.round(4).to_string())
argmax = pd.read_csv(C.RESULTS / 'argmax_rows.csv')
report = {'prereg_sha256': protocol['prereg_sha256'], 'bootstrap': 'paired match bootstrap, 1000 draws, seed 20261108, 95% percentile',
          'table': R.records(table), 'gates': gates, 'decision': decision, 'replication': R.records(replication),
          'gap_table': R.records(gap), 'exp4_gap_context': R.records(exp4_gap),
          'short_recall': R.records(recall_summary.reset_index()),
          'argmax': R.records(argmax.groupby(['label_set', 'view', 'variant'])[['spell_f1', 'count_error_ratio']].mean().reset_index()),
          'repro_check': C1.read_json(C.RESULTS / 'repro_check.json'),
          'reference_recheck': C1.read_json(C.RESULTS / 'reference_recheck.json')}
C1.write_json(C.RESULTS / 'proxy_report.json', report)

        arm  train_nll  heldout_nll_original  heldout_nll_target_like  gap_original  gap_target_like family  gap_growth_original  gap_growth_target_like  level flag_gap_growth_gt_25pct
exp2_recipe     0.1165                0.1884                   0.1913        0.0719           0.0747    NaN                0.000                  0.0000 family                    False
 exp5_touch     0.1237                0.1770                   0.1827        0.0533           0.0590    NaN               -0.259                 -0.2109 family                    False
exp_4 context (read in place):
        arm  train_nll  heldout_nll_original  heldout_nll_target_like  gap_original  gap_target_like family  gap_growth_vs_exp2_recipe  level flag_gap_growth_gt_25pct
exp2_recipe     0.1165                0.1884                   0.1913        0.0719           0.0747    NaN                     0.0000 family                    False
 exp4_touch     0.1237                0.1764                   0.1818        0.0

## Figure (proxy only)

In [8]:
order = [C.REFERENCE, 'exp4_touch_ens3', C.PRIMARY, 'exp45_touch_ens6']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label_set, marker in zip(C.S1_LABELS, ('o', 's', '^', 'v')):
    part = table[(table.label_set == label_set) & (table.view == 'target_like')].set_index('variant').reindex(order)
    axes[0].plot(order, part.mean_spell_f1, marker=marker, label=label_set)
    axes[1].plot(order, part.delta_vs_ref_ens3, marker=marker, label=label_set)
axes[0].set_ylabel('mean spellF1, target_like (7 held-out matches)'); axes[0].set_title('spellF1 by arm and label set')
axes[1].axhline(0, color='0.5', lw=0.8); axes[1].set_ylabel('Δ vs exp3_ref_ens3'); axes[1].set_title('paired Δ (point estimate)')
for ax in axes:
    ax.tick_params(axis='x', rotation=20); ax.legend(title='labels')
fig.tight_layout(); fig.savefig(C.RESULTS / 'fig_exp5.png', dpi=130)
print('saved', C.RESULTS / 'fig_exp5.png')

saved /home/jeremy-mboe/Documents/Kuliah/Sem5/barca/exp_jeremy/exp_5/results/fig_exp5.png


## Write guard: nothing under exp_0–exp_4 or `data/` changed since freezing

In [9]:
GUARD_FILE = 'write_guard_03.json'
before = C1.read_json(C.RESULTS / 'write_guard_before.json')
now = C.write_manifest()
guard = {name: {'unchanged': now[name] == state, 'files_before': state['files'], 'files_now': now[name]['files']}
         for name, state in before['folders'].items()}
C1.write_json(C.RESULTS / GUARD_FILE, {'checked_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
                                       'since': before['taken_at_utc'], 'folders': guard})
print('nothing written under exp_0-exp_4 / data/ since freezing:', all(g['unchanged'] for g in guard.values()), guard)


nothing written under exp_0-exp_4 / data/ since freezing: True {'exp_0': {'unchanged': True, 'files_before': 42630, 'files_now': 42630}, 'exp_1': {'unchanged': True, 'files_before': 215, 'files_now': 215}, 'exp_2': {'unchanged': True, 'files_before': 312, 'files_now': 312}, 'exp_3': {'unchanged': True, 'files_before': 407, 'files_now': 407}, 'exp_4': {'unchanged': True, 'files_before': 638, 'files_now': 638}, 'data': {'unchanged': True, 'files_before': 12, 'files_now': 12}}
